# RandomForest Cirrhosis Optimized
Target: Cirrhosis_Status

In [13]:
!pip install -q xgboost

In [30]:
!pip install catboost
# ==========================================================
# Liver Cirrhosis Prediction using CatBoost
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier



# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("/content/drive/MyDrive/cirrhosis.csv")

df.columns = df.columns.str.strip()


# Remove missing target

df = df.dropna(subset=["Stage"])


print("="*60)
print("Dataset Shape :", df.shape)
print("="*60)



# ==========================================================
# Features & Target
# ==========================================================

TARGET = "Stage"


X = df.drop(
    columns=["ID", TARGET],
    errors="ignore"
)


y = df[TARGET].astype(int)



print("\nTarget Classes:")
print(y.unique())



# ==========================================================
# Handle Missing Values
# ==========================================================

# CatBoost accepts missing numerical values
# For categorical values, replace NaN by string

cat_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()


for col in cat_cols:
    X[col] = X[col].fillna("Missing")



# ==========================================================
# Train/Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)



# ==========================================================
# CatBoost Model
# ==========================================================
from sklearn.neural_network import MLPClassifier


mlp_model = MLPClassifier(

    hidden_layer_sizes=(128,64),

    activation="relu",

    solver="adam",

    learning_rate_init=0.001,

    max_iter=500,

    random_state=42

)

# ==========================================================
# Training
# ==========================================================

print("\nTraining CatBoost...\n")


catboost_model.fit(

    X_train,

    y_train,

    cat_features=cat_cols,

    eval_set=(X_test,y_test),

    early_stopping_rounds=50

)



# ==========================================================
# Prediction
# ==========================================================

train_pred = catboost_model.predict(X_train)

test_pred = catboost_model.predict(X_test)



# Convert prediction shape

train_pred = train_pred.astype(int).flatten()

test_pred = test_pred.astype(int).flatten()



# ==========================================================
# Evaluation
# ==========================================================

print("="*60)


print(
    "Train Accuracy :",
    round(
        accuracy_score(
            y_train,
            train_pred
        )*100,
        2
    ),
    "%"
)


print(
    "Test Accuracy :",
    round(
        accuracy_score(
            y_test,
            test_pred
        )*100,
        2
    ),
    "%"
)


print("="*60)



print("\nClassification Report\n")


print(
    classification_report(
        y_test,
        test_pred
    )
)



print("\nConfusion Matrix\n")


print(
    confusion_matrix(
        y_test,
        test_pred
    )
)



# ==========================================================
# Cross Validation
# ==========================================================

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)


scores = cross_val_score(

    catboost_model,

    X,

    y,

    cv=cv,

    scoring="accuracy",

    params={
        "cat_features":cat_cols
    }

)


print("\nCross Validation Scores")

print(scores)



print(
    "\nMean Accuracy :",
    round(scores.mean()*100,2),
    "%"
)


print(
    "Standard Deviation :",
    round(scores.std()*100,2),
    "%"
)

Dataset Shape : (412, 20)

Target Classes:
[4 3 2 1]

Training CatBoost...

0:	learn: 0.4784422	test: 0.2114990	best: 0.2114990 (0)	total: 16.4ms	remaining: 8.19s
50:	learn: 0.6477387	test: 0.4598950	best: 0.4679777 (45)	total: 774ms	remaining: 6.82s
100:	learn: 0.6992939	test: 0.5085341	best: 0.5085341 (100)	total: 1.4s	remaining: 5.55s
150:	learn: 0.7538240	test: 0.4507482	best: 0.5085341 (100)	total: 2.18s	remaining: 5.03s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5085340694
bestIteration = 100

Shrink model to first 101 iterations.
Train Accuracy : 55.32 %
Test Accuracy : 40.96 %

Classification Report

              precision    recall  f1-score   support

           1       0.13      1.00      0.24         4
           2       0.33      0.16      0.21        19
           3       0.55      0.19      0.29        31
           4       0.64      0.72      0.68        29

    accuracy                           0.41        83
   macro avg       0.41      0.52

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
